# TokenTextSplitter

`TokenTextSplitter`는 텍스트를 **토큰 수**를 기준으로 분할합니다.

## 왜 토큰 기반 분할이 필요한가?

LLM(Large Language Model)은 **토큰 제한**이 있습니다. 예를 들어:
- GPT-3.5-turbo: 4,096 토큰
- GPT-4: 8,192 토큰 (일부 모델은 32K, 128K)
- Claude: 100K+ 토큰

문자 수 기반 분할(`CharacterTextSplitter`)은 정확한 토큰 수를 보장하지 못하므로, 토큰 제한을 준수해야 하는 경우 `TokenTextSplitter`가 필요합니다.

## 주요 특징

- **토큰 수 기반**: 문자 수가 아닌 토큰 수로 청크 크기 측정
- **정확한 제한 준수**: LLM의 토큰 제한을 초과하지 않도록 보장
- **다양한 토크나이저 지원**: tiktoken, NLTK, spaCy, HuggingFace 등

## 토큰이란?

토큰은 모델이 이해하는 텍스트의 최소 단위입니다.

- **영어**: 일반적으로 단어나 하위 단어 단위 (예: "playing" → ["play", "ing"])
- **한글**: 음절이나 형태소 단위로 분할되며, 영어보다 토큰 수가 많음

예시:
```
"Hello, World!" → 약 3-4 토큰
"안녕하세요, 세계!" → 약 6-8 토큰
```

## 사용 시나리오

- ✅ **LLM 토큰 제한 준수**: API 비용 관리 및 오류 방지
- ✅ **정확한 청크 크기**: 토큰 수를 정확히 제어해야 할 때
- ✅ **비용 최적화**: 토큰 사용량을 정밀하게 관리하여 API 비용 절감


# 03. TokenTextSplitter

## 학습 목표
- 문자(character) 수와 토큰(token) 수의 차이를 이해해요
- `TokenTextSplitter`로 LLM 토큰 제한을 정확히 준수하는 분할을 해요
- `tiktoken`으로 실제 토큰 수를 계산해서 두 방식을 비교해요

## 사전 지식
- 02-RecursiveCharacterTextSplitter 완료
- 토큰(token) 개념: LLM이 이해하는 텍스트의 최소 단위
- `pip install tiktoken` 설치

---

> **왜 토큰 기반 분할이 필요한가?**
>
> `RecursiveCharacterTextSplitter`는 **문자 수**로 크기를 측정해요. 그런데 LLM API는 **토큰 수**로 요금을 청구하고 제한을 걸어요. 한글 1글자는 보통 2~3토큰이라서 문자 수로만 관리하면 토큰 제한을 초과할 수 있어요.


> 🔑 **핵심 개념**: 한글 1글자 = 약 1.5~2 토큰입니다. 문자 수 기반 분할기로 `chunk_size=500`자를 설정하면 실제로는 750~1000토큰이 되어 LLM API 제한을 초과할 수 있습니다. 토큰 제한이 엄격한 프로젝트에서는 `TokenTextSplitter`를 사용하세요.

## 샘플 데이터 로드


In [1]:
# 샘플 데이터 파일을 읽어옵니다.
with open("./data/appendix-keywords.txt", encoding="utf-8") as f:
    file = f.read()

print(f"전체 텍스트 길이: {len(file)} 문자")
print(f"\n처음 300자:")
print(file[:300])


전체 텍스트 길이: 5733 문자

처음 300자:
Semantic Search

정의: 의미론적 검색은 사용자의 질의를 단순한 키워드 매칭을 넘어서 그 의미를 파악하여 관련된 결과를 반환하는 검색 방식입니다.
예시: 사용자가 "태양계 행성"이라고 검색하면, "목성", "화성" 등과 같이 관련된 행성에 대한 정보를 반환합니다.
연관키워드: 자연어 처리, 검색 알고리즘, 데이터 마이닝

Embedding

정의: 임베딩은 단어나 문장 같은 텍스트 데이터를 저차원의 연속적인 벡터로 변환하는 과정입니다. 이를 통해 컴퓨터가 텍스트를 이해하고 처리할 수 있게 합니다.
예시: "사과"라는 단


## 1. tiktoken 기반 분할

OpenAI의 `tiktoken`은 빠른 BPE(Byte Pair Encoding) 토크나이저입니다. GPT 모델과 동일한 방식으로 토큰을 계산합니다.

### CharacterTextSplitter with tiktoken

`CharacterTextSplitter`의 `from_tiktoken_encoder()` 메서드를 사용하면 토큰 수를 기준으로 분할할 수 있습니다.


> ⚠️ **자주 하는 실수**: `CharacterTextSplitter.from_tiktoken_encoder()`는 tiktoken으로 토큰 수를 측정하지만, 내부적으로 CharacterTextSplitter로 먼저 분할하기 때문에 일부 청크가 `chunk_size`를 초과할 수 있습니다. 엄격한 제한이 필요하면 `TokenTextSplitter`를 사용하세요.

In [2]:
# ============================================================
# TODO: CharacterTextSplitter.from_tiktoken_encoder()로 토큰 기반 분할을 해보세요
# 힌트: CharacterTextSplitter.from_tiktoken_encoder(chunk_size=300, chunk_overlap=0)
# 예상 결과: 분할된 청크 개수와 첫 번째 청크 내용이 출력됩니다
# ============================================================

from langchain_text_splitters import CharacterTextSplitter

# 1단계: tiktoken 인코더 기반 Splitter 생성
# - from_tiktoken_encoder(): tiktoken 토크나이저로 크기를 측정하는 Splitter 반환
# - chunk_size=300: 최대 300 토큰 (문자 수 아님)
text_splitter = CharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=300,  # 토큰 수: 300
    chunk_overlap=0,  # 중복 없음
)

# 2단계: 텍스트 분할
texts = text_splitter.split_text(file)

print(f"분할된 청크 개수: {len(texts)}")
print(f"\n첫 번째 청크:")
print("=" * 60)
print(texts[0])
print(f"\n첫 번째 청크 문자 수: {len(texts[0])}")

Created a chunk of size 358, which is longer than the specified 300
Created a chunk of size 315, which is longer than the specified 300
Created a chunk of size 305, which is longer than the specified 300
Created a chunk of size 366, which is longer than the specified 300
Created a chunk of size 330, which is longer than the specified 300
Created a chunk of size 351, which is longer than the specified 300
Created a chunk of size 378, which is longer than the specified 300
Created a chunk of size 361, which is longer than the specified 300
Created a chunk of size 350, which is longer than the specified 300
Created a chunk of size 362, which is longer than the specified 300
Created a chunk of size 335, which is longer than the specified 300
Created a chunk of size 353, which is longer than the specified 300
Created a chunk of size 358, which is longer than the specified 300
Created a chunk of size 336, which is longer than the specified 300
Created a chunk of size 324, which is longer tha

분할된 청크 개수: 51

첫 번째 청크:
Semantic Search

첫 번째 청크 문자 수: 15


**참고**: `CharacterTextSplitter.from_tiktoken_encoder()`는 먼저 `CharacterTextSplitter`로 분할한 후, tiktoken으로 토큰 수를 측정하여 병합합니다. 따라서 일부 청크가 `chunk_size`보다 클 수 있습니다.

더 정확한 토큰 제한을 원한다면 `RecursiveCharacterTextSplitter.from_tiktoken_encoder()`나 `TokenTextSplitter`를 사용하세요.


## 2. TokenTextSplitter

`TokenTextSplitter`는 순수하게 토큰 단위로 텍스트를 분할합니다. 각 청크가 지정된 토큰 수를 초과하지 않도록 보장합니다.


In [3]:
# ============================================================
# TODO: TokenTextSplitter로 텍스트를 토큰 단위로 분할해보세요
# 힌트: TokenTextSplitter(chunk_size=300, chunk_overlap=50) → split_text(file)
# 예상 결과: 모든 청크의 토큰 수가 300 이하로 분할됩니다
# ============================================================

from langchain_text_splitters import TokenTextSplitter

# 1단계: TokenTextSplitter 생성
# - TokenTextSplitter: 토큰 단위로 정확하게 분할 (from_tiktoken_encoder보다 엄격)
# - chunk_size=300: 최대 300 토큰
# - chunk_overlap=50: 50 토큰 중복
token_splitter = TokenTextSplitter(
    chunk_size=300,  # 최대 300 토큰
    chunk_overlap=50,  # 50 토큰 중복
)

# 2단계: 텍스트 분할
token_texts = token_splitter.split_text(file)

print(f"TokenTextSplitter 청크 개수: {len(token_texts)}")
print(f"\n첫 번째 청크:")
print("=" * 60)
print(token_texts[0])
print(f"\n첫 번째 청크 문자 수: {len(token_texts[0])}")

TokenTextSplitter 청크 개수: 42

첫 번째 청크:
Semantic Search

정의: 의미론적 검색은 사용자의 질의를 단순한 키워드 매칭을 넘어서 그 의미를 파악하여 관련된 결과를 반환하는 검색 방식입니다.
예시: 사용자가 "태양계 행성"이라고 검색하면, "목성", "화성" 등과 같이 관련된 행성에 대한 정보를 반환합니다.
연�

첫 번째 청크 문자 수: 157


> 🎯 **강의 포인트**: `tiktoken.get_encoding("cl100k_base")`는 GPT-4, GPT-3.5-turbo가 사용하는 인코딩입니다. `len(encoding.encode(text))`로 실제 토큰 수를 직접 계산해서 청크 설정을 검증하는 습관을 들이세요.

## 3. 문자 수 vs 토큰 수 비교

동일한 텍스트를 문자 수 기반과 토큰 수 기반으로 분할하여 차이를 비교합니다.


In [4]:
# ============================================================
# TODO: RecursiveCharacterTextSplitter(문자 기반)와 TokenTextSplitter(토큰 기반)의 결과를 비교해보세요
# 힌트: tiktoken.get_encoding("cl100k_base")로 각 청크의 실제 토큰 수를 계산하세요
# 예상 결과: 토큰 수 기반 분할은 토큰 제한 초과 청크가 0개임을 확인합니다
# ============================================================

from langchain_text_splitters import RecursiveCharacterTextSplitter
import tiktoken

# 1단계: 문자 수 기반 분할
char_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1200,  # 문자 수: 1200
    chunk_overlap=200,
)
char_chunks = char_splitter.split_text(file)

# 2단계: 토큰 수 기반 분할
token_splitter = TokenTextSplitter(
    chunk_size=300,  # 토큰 수: 300
    chunk_overlap=50,
)
token_chunks = token_splitter.split_text(file)

# 3단계: tiktoken 인코더로 실제 토큰 수 계산
# - cl100k_base: GPT-4, gpt-3.5-turbo 등이 사용하는 인코딩
encoding = tiktoken.get_encoding("cl100k_base")

print("=" * 60)
print("📊 문자 수 기반 vs 토큰 수 기반 비교")
print("=" * 60)
print(f"\n문자 수 기반 (RecursiveCharacterTextSplitter):")
print(f"  - 청크 개수: {len(char_chunks)}")
print(f"  - 평균 문자 수: {sum(len(c) for c in char_chunks) / len(char_chunks):.1f}")
char_token_counts = [len(encoding.encode(c)) for c in char_chunks]
print(f"  - 평균 토큰 수: {sum(char_token_counts) / len(char_token_counts):.1f}")
print(f"  - 최대 토큰 수: {max(char_token_counts)}")

print(f"\n토큰 수 기반 (TokenTextSplitter):")
print(f"  - 청크 개수: {len(token_chunks)}")
print(f"  - 평균 문자 수: {sum(len(c) for c in token_chunks) / len(token_chunks):.1f}")
token_token_counts = [len(encoding.encode(c)) for c in token_chunks]
print(f"  - 평균 토큰 수: {sum(token_token_counts) / len(token_token_counts):.1f}")
print(f"  - 최대 토큰 수: {max(token_token_counts)}")

print(f"\n💡 토큰 제한 300으로 설정 시:")
print(f"  - 문자 수 기반: {sum(1 for c in char_token_counts if c > 300)}개 청크가 제한 초과")
print(f"  - 토큰 수 기반: {sum(1 for c in token_token_counts if c > 300)}개 청크가 제한 초과")

📊 문자 수 기반 vs 토큰 수 기반 비교

문자 수 기반 (RecursiveCharacterTextSplitter):
  - 청크 개수: 6
  - 평균 문자 수: 1052.8
  - 평균 토큰 수: 813.2
  - 최대 토큰 수: 917

토큰 수 기반 (TokenTextSplitter):
  - 청크 개수: 42
  - 평균 문자 수: 164.0
  - 평균 토큰 수: 127.4
  - 최대 토큰 수: 147

💡 토큰 제한 300으로 설정 시:
  - 문자 수 기반: 6개 청크가 제한 초과
  - 토큰 수 기반: 0개 청크가 제한 초과


> 💡 **실무 팁**: OpenAI API 비용 절감을 위해 `chunk_size`를 작게 설정하고 싶다면, 먼저 `tiktoken`으로 문서의 평균 토큰 수를 계산한 후 `chunk_size`를 결정하세요. 이 과정 없이 임의로 설정하면 정보 손실이나 비용 낭비가 생깁니다.

## 4. 한글 처리: KoNLPy (선택)

한글 텍스트의 경우 영어와 다른 토큰화 방식이 필요할 수 있습니다. `KoNLPy`는 한국어 자연어 처리를 위한 패키지입니다.

**참고**: 아래 예제는 선택사항입니다. 일반적으로 tiktoken이나 TokenTextSplitter로 충분합니다.


### KoNLPy 설치 (선택)

KoNLPy를 사용하려면 먼저 설치가 필요합니다.

```bash
pip install konlpy
```

**주의**: KoNLPy는 Java가 필요하며, 초기 로딩이 느릴 수 있습니다. 일반적인 경우 TokenTextSplitter나 tiktoken으로 충분합니다.


## 핵심 정리 및 마무리

### 문자 수 vs 토큰 수

| 특징 | 문자 수 기반 | 토큰 수 기반 |
|------|:-----------:|:-----------:|
| 측정 단위 | `len()` | tiktoken 기준 |
| 정확성 | 토큰 수 예측 어려움 | 정확한 토큰 수 |
| LLM 제한 준수 | 보장 안 됨 | 정확히 준수 |
| 속도 | 빠름 | 약간 느림 |

### 한글 토큰 예측 팁
> 한글은 평균 1글자당 1.5~2토큰이에요. `chunk_size=300`(토큰)으로 설정하면 실제로는 약 150~200자 분량의 한글 텍스트가 들어가요. 이 점을 감안해서 청크 크기를 설정해야 해요.

---

## 다음 예고

문자/토큰 기반의 **크기(size)** 분할에서 **의미(semantic)** 기반 분할로 넘어가요.

- **04-SemanticChunker**: 임베딩 유사도 차이를 분석해서 의미 단위로 분할
